# 🌊 AquaSentinel AI: Production Training & Evaluation Pipeline
### Multi-Phase Side-Scan Sonar Segmentation (GhostVision + SSS-Mine / NOMBO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaghavKacker/Aqua-Sentinel/blob/main/notebooks/AquaSentinel_Training.ipynb)

This notebook follows the 11-step project specification to inspect real datasets, normalize annotations, train a YOLO-Seg segmentation model on Google Colab GPU, and evaluate performance with acoustic shadow verification.

```text
1. Mount Google Drive               [DONE]
2. Install dependencies             [DONE]
3. Download GhostVision             [DONE]
4. Download SSS-Mine                [DONE]
5. Inspect both datasets            ← WE ARE HERE (Running diagnostics)
6. Convert annotations (YOLO-Seg)   [Next step]
7. Create train / val / test        [Mission-isolated split]
8. Train YOLO-Seg                   [yolo11n-seg transfer learning]
9. Validate                         [Precision, Recall, mAP50, mAP50-95]
10. Test                            [Unseen survey evaluation]
11. Compare Results                 [YOLO vs YOLO + Acoustic Gate]
```

---

## Phase 1: Mount Google Drive
Connects Google Drive where your datasets (`/content/drive/MyDrive/AquaSentinel/datasets`) and checkpoints are stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## Phase 2: Install Dependencies
Installs Ultralytics, OpenCV headless, and tools for hydrographic dataset parsing.

In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib tabulate

import torch, ultralytics
print(f"\nPyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
ultralytics.checks()

## Phase 3 & 4: Locate Downloaded Datasets
Points to the confirmed dataset directories in Google Drive (`GhostVision` and `SSS-Mine`).

In [ ]:
import os
from pathlib import Path

# Primary dataset directory in Google Drive
DATASET_DIR = "/content/drive/MyDrive/AquaSentinel/datasets"
ghostvision_path = os.path.join(DATASET_DIR, "GhostVision")
sss_mine_path = os.path.join(DATASET_DIR, "SSS-Mine")

# Fallback search if folder was placed elsewhere
def resolve_path(primary, fallbacks):
    if os.path.exists(primary):
        return primary
    for f in fallbacks:
        if os.path.exists(f):
            return f
    return primary

ghostvision_path = resolve_path(ghostvision_path, [
    "/content/GhostVision", "./GhostVision", "./data/GhostVision",
    "/content/drive/MyDrive/GhostVision"
])

sss_mine_path = resolve_path(sss_mine_path, [
    "/content/SSS-Mine", "./SSS-Mine", "./data/SSS-Mine",
    "/content/drive/MyDrive/SSS-Mine"
])

print("=" * 70)
print(f"GhostVision Path: {ghostvision_path} (Exists: {os.path.exists(ghostvision_path)})")
print(f"SSS-Mine Path:    {sss_mine_path} (Exists: {os.path.exists(sss_mine_path)})")
print("=" * 70)

## Phase 5: Deep Inspection of Both Datasets (CURRENT STEP)
### Step 5.1: Directory Tree & File Extension Audit
Audits directory hierarchy, image file counts, and file types across GhostVision and SSS-Mine.

In [ ]:
from collections import Counter
import os

def inspect_dataset(dataset_path, max_depth=3):
    print("=" * 80)
    print(f"DATASET: {dataset_path}")
    print("=" * 80)

    if not os.path.exists(dataset_path):
        print("❌ Dataset folder does not exist!")
        return

    extension_counts = Counter()
    total_files = 0

    for root, dirs, files in os.walk(dataset_path):
        relative = os.path.relpath(root, dataset_path)
        depth = 0 if relative == "." else relative.count(os.sep) + 1

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth
        folder_name = os.path.basename(root)

        print(f"{indent}📁 {folder_name}/")

        for file in files[:15]:
            print(f"{indent}  📄 {file}")

        if len(files) > 15:
            print(f"{indent}  ... +{len(files)-15} more files")

        for file in files:
            total_files += 1
            ext = os.path.splitext(file)[1].lower()
            if ext:
                extension_counts[ext] += 1
            else:
                extension_counts["[no extension]"] += 1

    print("\nFile summary:")
    print("Total files:", total_files)
    print("Extensions:")
    for ext, count in extension_counts.most_common():
        print(f"  {ext}: {count}")
    print()

# Run tree inspection on both datasets
inspect_dataset(ghostvision_path)
inspect_dataset(sss_mine_path)

### Step 5.2: Annotation Inspection & Class Discovery
This diagnostic inspects:
1. **GhostVision:** Finds all `.jsonl`, `.json`, `.txt`, `.csv` annotation files and prints label keys/categories.
2. **SSS-Mine:** Reads the `.txt` label files across survey years (`2010`, `2015`, `2017`, `2018`, `2021`) and discovers unique class IDs and coordinate format.

In [ ]:
import json, glob
from collections import Counter
from pathlib import Path

print("=" * 80)
print("INSPECTING GHOSTVISION ANNOTATIONS")
print("=" * 80)

gv_dir = Path(ghostvision_path)
# Find all non-image metadata / annotation files
gv_anno_files = []
for pattern in ['**/*.jsonl', '**/*.json', '**/*.csv', '**/*.yaml']:
    for p in gv_dir.glob(pattern):
        # Skip internal cache lock files
        if '.cache' not in str(p) or p.name.endswith('.jsonl') or p.name.endswith('.json'):
            gv_anno_files.append(p)

print(f"Found {len(gv_anno_files)} candidate annotation/metadata files in GhostVision:")
for f in gv_anno_files[:10]:
    print(f"  📄 {f.relative_to(gv_dir)} ({f.stat().st_size} bytes)")

# Inspect JSON/JSONL contents
for f in gv_anno_files:
    if f.suffix == '.jsonl':
        print(f"\nInspecting JSONL: {f.name} (first 2 lines):")
        try:
            with open(f, 'r', encoding='utf-8') as jlf:
                for i in range(2):
                    line = jlf.readline()
                    if line:
                        sample_obj = json.loads(line)
                        keys = list(sample_obj.keys())
                        print(f"  Row {i+1} keys: {keys}")
                        # Show label/category fields if present
                        for k in ['label', 'labels', 'objects', 'annotations', 'category', 'tag']:
                            if k in sample_obj:
                                print(f"    {k}: {sample_obj[k]}")
        except Exception as e:
            print(f"  [Error reading {f.name}: {e}]")
    elif f.suffix == '.json' and not f.name.startswith('.'):
        print(f"\nInspecting JSON: {f.name}:")
        try:
            with open(f, 'r', encoding='utf-8') as jf:
                data = json.load(jf)
                if isinstance(data, dict):
                    print(f"  Top-level keys: {list(data.keys())}")
                    if 'categories' in data:
                        print(f"  Categories: {data['categories']}")
                elif isinstance(data, list) and data:
                    print(f"  List of {len(data)} items. First item keys: {list(data[0].keys()) if isinstance(data[0], dict) else data[0]}")
        except Exception as e:
            print(f"  [Error reading {f.name}: {e}]")

print("\n" + "=" * 80)
print("INSPECTING SSS-MINE / NOMBO ANNOTATIONS")
print("=" * 80)

mine_dir = Path(sss_mine_path)
mine_txt_files = list(mine_dir.glob('**/*.txt'))
print(f"Found {len(mine_txt_files)} TXT annotation files in SSS-Mine.")

mine_class_counter = Counter()
sample_lines = []

for tf in mine_txt_files:
    try:
        with open(tf, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = parts[0]
                    mine_class_counter[cls_id] += 1
                    if len(sample_lines) < 5:
                        sample_lines.append((tf.name, line.strip()))
    except Exception:
        pass

print(f"\nSSS-Mine Discovered Classes (Class ID -> Count): {dict(mine_class_counter)}")
print("\nSample SSS-Mine TXT Rows:")
for fname, l in sample_lines:
    print(f"  {fname}: {l}")

### Step 5.3: Visual Sonar Verification
Visualizes raw sonar waterfall crops from GhostVision and SSS-Mine to verify acoustic highlight and shadow morphology.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Collect sample images
gv_imgs = list(Path(ghostvision_path).glob('**/*.jpg'))[:2]
mine_imgs = list(Path(sss_mine_path).glob('**/*.jpg'))[:2]
samples = [('GhostVision', p) for p in gv_imgs] + [('SSS-Mine', p) for p in mine_imgs]

if samples:
    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
    if len(samples) == 1:
        axes = [axes]
    for i, (ds_name, img_p) in enumerate(samples):
        img = cv2.imread(str(img_p))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            axes[i].imshow(img_rgb)
            axes[i].set_title(f"[{ds_name}]\n{img_p.parent.name}/{img_p.name}\n({w}x{h})", fontsize=9)
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No sample images found to plot. Please check paths.")

## Phase 6: Convert Annotations (Planned Next Step)
Converts the inspected labels into normalized Ultralytics YOLO-Seg polygon segmentation format.
Once Step 5 outputs the exact categories from GhostVision and SSS-Mine, run this cell to map them into the canonical schema:

| Class ID | Canonical Label | Source |
| :--- | :--- | :--- |
| `0` | `crab_pot` | GhostVision (traps, pots) |
| `1` | `ghost_gear` | GhostVision (derelict nets, lines) |
| `2` | `mine_cylinder` | SSS-Mine (mine-like targets) |
| `3` | `debris_anomaly` | Unclassified sonar anomalies |

In [ ]:
print("Ready for Phase 6 (Convert Annotations).")
print("After running Step 5, paste the discovered classes and we will finalize the conversion logic!")

## Phase 7: Create Survey-Isolated Train / Val / Test Splits
Groups tiles by survey mission/year (not random image splitting) to prevent ping leakage across splits.

In [ ]:
print("Ready for Phase 7 (Survey/Mission Splitting & dataset.yaml generation).")

## Phase 8: Train YOLO-Seg Model on Colab GPU
Transfers pre-trained weights from `yolo11n-seg.pt` with acoustic-optimized augmentations (`fliplr=0.5`, `flipud=0.0`).

In [ ]:
print("Ready for Phase 8 (Model Training with yolo11n-seg).")

## Phase 9 & 10: Validate & Test on Unseen Mission
Evaluates mAP50, mAP50-95, box precision, mask precision, and recall on the unseen test split.

In [ ]:
print("Ready for Phase 9 & 10 (Validation & Unseen Mission Testing).")

## Phase 11: Compare Results & Export
Compares pure YOLO-Seg predictions against YOLO-Seg + Acoustic Shadow Verification, then exports `best.pt` for local deployment.

In [ ]:
print("Ready for Phase 11 (Comparative Benchmark & best.pt Export).")